# Run national version of the NHP model

In [ ]:
## Uncomment if running notebook directly
# %cd ..
# %pip install .

In [0]:
import gzip
import uuid
import multiprocessing as mp
import os
import json

from datetime import datetime

from azure.identity import ManagedIdentityCredential
from azure.storage.blob import BlobServiceClient
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from azure.data.tables import TableServiceClient

from nhp.databricks.national import DatabricksNational
from nhp.model.params import load_params
from nhp.model.run import run_all

In [ ]:
mp.set_start_method("spawn", force=True)

In [ ]:
spark = DatabricksSession.builder.getOrCreate()
w = WorkspaceClient()
dbutils = w.dbutils
model_run_id = str(uuid.uuid4())

## Get the notebook parameters

In [0]:
params_path = dbutils.widgets.get("params_path")

data_path  = dbutils.widgets.get("data_path")
save_full_model_results = dbutils.widgets.get("save_full_model_results") == "True"

## Load the model run parameters

In [0]:
params = load_params(params_path)

outputs_version = params["app_version"]

create_datetime = datetime.now()
params["create_datetime"] = f"{create_datetime:%Y%m%d_%H%M%S}"

metadata = {
    k: v
    for k, v in params.items()
    if not isinstance(v, dict) and not isinstance(v, list)
}
metadata

## Set up the data

In [0]:
nhp_data = DatabricksNational.create(spark, data_path, sample_rate = 0.01, seed = params["seed"])

## Run the model

In [0]:
model_run_start_time = datetime.now()
results, variants = run_all(params, nhp_data, save_full_model_results=save_full_model_results)
model_run_end_time = datetime.now()
elapsed_time = model_run_end_time - model_run_start_time

In [ ]:
run_metadata = {
    "model_run_start_time": model_run_start_time.isoformat(), 
    "model_run_elapsed_time_seconds": elapsed_time.total_seconds(), 
    "model_run_end_time": model_run_end_time.isoformat(),
    "viewable": False,
    "status": "complete",
    }

## Upload results

In [ ]:
blob_client = BlobServiceClient("https://nhpsa.blob.core.windows.net", credential=ManagedIdentityCredential())
cont = blob_client.get_container_client("results")

### Upload compressed JSON results

Should be removed in future versions of the model

In [ ]:
from nhp.model.results import generate_results_json

results_file = generate_results_json(results, params, variants)
results_json_gz_path = f"prod/{params['app_version']}/{results_file}.json.gz"
with open(f"results/{results_file}.json", "rb") as file:
    cont.upload_blob(
        results_json_gz_path,
        gzip.compress(file.read()),
        metadata={k: str(v) for k, v in metadata.items()},
        overwrite=True,
    )

run_metadata["results_json_gz_path"] = results_json_gz_path

# make sure to remove the file from databricks storage [in the asset bundle deployment]
os.unlink(f"results/{results_file}.json")

### Upload the parquet files

In [ ]:
file_path = "/".join(
    [
        "aggregated-model-results",
        params["app_version"],
        params["dataset"],
        params["scenario"],
        params["create_datetime"],
    ]
)

# Upload parquet files
for k, v in results.items():
    cont.upload_blob(
        file_path + f"/{k}.parquet",
        v.to_parquet(index=False),
        overwrite=True,
        metadata={k:str(v) for k, v in metadata.items()},
    )

# Upload params and variants
cont.upload_blob(
    f"{file_path}/params.json",
    json.dumps(params).encode("utf-8"),
    overwrite=True,
    metadata={k:str(v) for k, v in metadata.items()},
)
cont.upload_blob(
    f"{file_path}/variants.json",
    json.dumps(variants).encode("utf-8"),
    overwrite=True,
    metadata={k:str(v) for k, v in metadata.items()},
)

run_metadata["aggregated_results_path"] = file_path

### Save the full model results

In [ ]:
if save_full_model_results:
    from pathlib import Path

    # Save the IP full model results to storage
    # From docker_run._upload_full_model_results

    path = Path(f"results/{params["dataset"]}/{params["scenario"]}/{params["create_datetime"]}")
    for file in path.glob("**/*.parquet"):
        filename = file.as_posix()[8:]
        with open(file, "rb") as f:
            cont.upload_blob(
                f"full-model-results/{outputs_version}/{filename}",
                f.read(),
                overwrite=True,
            )
        # make sure to remove the file from databricks storage [in the asset bundle deployment]
        os.unlink(file)
    run_metadata["save_full_model_results"] = True
else:
    run_metadata["save_full_model_results"] = False

## Update Azure Table Storage

In [ ]:
# Compile metadata
entity = {
        "PartitionKey": params["dataset"],
        "RowKey": model_run_id        
    } | metadata

entity = entity | run_metadata

entity["outputs_app_uri"] = f"{params['dataset']}/{model_run_id}"

entity["create_datetime"] = create_datetime

In [ ]:
# Upload to ATS
secret_scope = "nhp_model_databricks"
account_name = dbutils.secrets.get(secret_scope, "account_name").strip()
table_name = dbutils.secrets.get(secret_scope, "table_name").strip()

## Uncomment if running notebook directly
# table_sas = dbutils.secrets.get(secret_scope, "table_sas").strip().lstrip("?")
# account_url = (
#     f"https://{account_name}.table.core.windows.net"
#     f"?{table_sas}"
# )
# service_client = TableServiceClient(endpoint=account_url)

service_client = TableServiceClient(
    f"https://{account_name}.table.core.windows.net", 
    credential=ManagedIdentityCredential())

table_client = service_client.get_table_client(table_name)

table_client.create_entity(entity=entity)

## Remove the params file from the queue

In [ ]:
# remove the file from the queue
os.unlink(params_path)